In [ ]:
# Cell 1: Import required libraries
import os
import tempfile
from typing import List, Dict, Any
import streamlit as st
from dotenv import load_dotenv

# LangChain imports
from langchain.document_loaders import TextLoader, PyPDFLoader, CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.llms import HuggingFacePipeline
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# Load environment variables (if you have any)
load_dotenv()

In [ ]:
# Cell 2: Create the core chatbot class
class LightweightRAGChatbot:
    """
    A lightweight RAG chatbot that runs efficiently on CPU
    """
    
    def __init__(self):
        self.embeddings = None
        self.vectorstore = None
        self.chain = None
        self.memory = None
        self.initialize_components()
    
    def initialize_components(self):
        """Initialize embedding model and other components"""
        try:
            # Use a lightweight embedding model
            self.embeddings = HuggingFaceEmbeddings(
                model_name="sentence-transformers/all-MiniLM-L6-v2",
                model_kwargs={'device': 'cpu'},
                encode_kwargs={'normalize_embeddings': True}
            )
            print("✅ Embeddings model loaded successfully!")
        except Exception as e:
            print(f"❌ Error loading embeddings: {e}")
            raise
    
    def create_vectorstore(self, documents):
        """Create a vector store from documents"""
        try:
            # Split documents into chunks
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=500,
                chunk_overlap=50,
                length_function=len,
                separators=["\n\n", "\n", " ", ""]
            )
            
            chunks = text_splitter.split_documents(documents)
            print(f"✅ Created {len(chunks)} document chunks")
            
            # Create vector store
            self.vectorstore = Chroma.from_documents(
                documents=chunks,
                embedding=self.embeddings,
                persist_directory="./chroma_db"  # Local persistence
            )
            print("✅ Vector store created successfully!")
            return True
            
        except Exception as e:
            print(f"❌ Error creating vectorstore: {e}")
            return False
    
    def create_conversation_chain(self):
        """Create the conversational retrieval chain"""
        if not self.vectorstore:
            print("❌ Vector store not initialized!")
            return False
        
        try:
            # Initialize memory for conversation context
            self.memory = ConversationBufferMemory(
                memory_key="chat_history",
                return_messages=True,
                output_key='answer'
            )
            
            # For CPU-friendly implementation, we'll use a template-based approach
            # without loading a heavy LLM locally
            from langchain.prompts import PromptTemplate
            from langchain.chains import LLMChain
            
            # Define a simple prompt template
            template = """You are a helpful assistant. Use the following context to answer the user's question.
            If you don't know the answer, just say you don't know. Don't try to make up an answer.
            
            Context: {context}
            
            Chat History: {chat_history}
            
            Question: {question}
            
            Answer: Let me help you with that based on the available information."""
            
            prompt = PromptTemplate(
                input_variables=["context", "chat_history", "question"],
                template=template
            )
            
            # For demo purposes, we'll use a mock LLM
            # In production, you can replace this with a lightweight local LLM or API
            from langchain.llms.fake import FakeListLLM
            
            responses = [
                "Based on the context provided, I can help you with that. " +
                "The relevant information shows that this is related to the documents you've provided.",
                "From the available information in your documents, I can tell you that this is an important topic.",
                "According to the context, this is something that is covered in your knowledge base."
            ]
            
            self.llm = FakeListLLM(responses=responses)
            
            # Create the chain
            self.chain = ConversationalRetrievalChain.from_llm(
                llm=self.llm,
                retriever=self.vectorstore.as_retriever(search_kwargs={"k": 3}),
                memory=self.memory,
                combine_docs_chain_kwargs={"prompt": prompt}
            )
            
            print("✅ Conversation chain created successfully!")
            return True
            
        except Exception as e:
            print(f"❌ Error creating conversation chain: {e}")
            return False
    
    def chat(self, message: str) -> str:
        """Process a chat message and return response"""
        if not self.chain:
            return "Chatbot not properly initialized. Please check the setup."
        
        try:
            response = self.chain({"question": message})
            return response['answer']
        except Exception as e:
            return f"Error processing message: {str(e)}"
    
    def clear_memory(self):
        """Clear conversation memory"""
        if self.memory:
            self.memory.clear()
            print("✅ Conversation memory cleared!")